In [ ]:
import os
from pathlib import Path
from types import SimpleNamespace

import numpy as np

try:
    from scipy import ndimage, signal, optimize, interpolate, stats
except Exception:
    ndimage = signal = optimize = interpolate = stats = None

def n_elements(x):
    if x is None:
        return 0
    try:
        return np.size(x)
    except Exception:
        return 1

def read_parameter_file(parfile):
    params = {}
    path = Path(parfile)
    if not path.exists():
        raise FileNotFoundError(parfile)

    for raw in path.read_text().splitlines():
        line = raw.split(';', 1)[0].strip()
        if not line or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip().lower()
        value = value.strip()
        value = value.replace('[', 'np.array([').replace(']', '])')
        value = value.replace('^', '**')
        try:
            params[key] = eval(value, {'np': np, 'array': np.array})
        except Exception:
            params[key] = value.strip("'\"")
    return params

def write_idl_array_line(f, name, arr, comment=''):
    arr = np.asarray(arr).ravel()
    values = ','.join(f'{v:10.4E}' if abs(v) >= 1e4 or (abs(v) < 1e-3 and v != 0) else f'{v:10.4f}' for v in arr)
    f.write(f'{name.upper()}=[{values}]')
    if comment:
        f.write(f' ; {comment}')
    f.write('\n')

def robust_sigma(values):
    values = np.asarray(values, dtype=float)
    med = np.nanmedian(values)
    return 1.4826 * np.nanmedian(np.abs(values - med))

def linear_interp(x, y, x_new):
    return np.interp(x_new, np.asarray(x, dtype=float), np.asarray(y, dtype=float))

def congrid(array, new_shape):
    array = np.asarray(array, dtype=float)
    if ndimage is None:
        raise ImportError('scipy is required for congrid')
    zoom = [n / o for n, o in zip(new_shape, array.shape)]
    return ndimage.zoom(array, zoom, order=1)

def idl_hist2d(x, y, xbin, ybin, xmin, xmax, ymin, ymax):
    x_edges = np.arange(xmin, xmax + xbin, xbin)
    y_edges = np.arange(ymin, ymax + ybin, ybin)
    hist, _, _ = np.histogram2d(x, y, bins=[x_edges, y_edges])
    return hist

def load_model_file(path):
    with open(path, 'r') as f:
        age_line = f.readline().strip()
        feh_line = f.readline().strip()
        shape_line = f.readline().strip()
        shape = tuple(int(v) for v in shape_line.replace(',', ' ').split()[:2])
        data = np.loadtxt(f)
    return age_line, feh_line, data.reshape(shape)

def save_model_file(path, age_label, feh_label, model):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    model = np.asarray(model, dtype=float)
    with open(path, 'w') as f:
        f.write(age_label + '\n')
        f.write(feh_label + '\n')
        f.write(f'{model.shape[0]} {model.shape[1]}\n')
        np.savetxt(f, model)


In [ ]:
def sample_power_law_mass(rng, min_mass, max_mass, imf_slope):
    max_mf = min_mass ** (-imf_slope)
    while True:
        mt = rng.random() * (max_mass - min_mass) + min_mass
        if rng.random() < (mt ** (-imf_slope)) / max_mf:
            return mt

def mkcmd_opt(parin, fileout, readisoc, seed=None):
    p = read_parameter_file(parin)
    rng = np.random.default_rng(seed)
    ages = np.asarray(p['ages'], dtype=float)
    sfh = np.asarray(p['sfh'], dtype=float)
    feh = np.asarray(p['feh'], dtype=float)
    sfh = sfh / sfh.sum()
    imf = float(p['imf'])
    dmod = float(p.get('dmod', 0.0))
    a_k = float(p.get('a_k', 0.0))
    sigred = float(p.get('sigred', 0.0))
    npix = int(p['npix'])
    maxstar = int(p.get('maxstar', 1_000_000))

    isos, minmass, maxmass = [], [], []
    for age, z in zip(ages, feh):
        iso = readisoc('girardi', age, z)
        isos.append(iso)
        mass = np.asarray(iso.i_mass, dtype=float)
        minmass.append(max(float(p.get('minmass', mass.min())), mass.min()))
        maxmass.append(mass.max())
    minmass = np.asarray(minmass)
    maxmass = np.asarray(maxmass)

    totlum = float(p.get('totlum', p.get('maxstar', 10000)))
    lum1 = 0.0
    count = 0
    rows = []

    while lum1 < totlum and count < maxstar:
        age_index = rng.choice(len(sfh), p=sfh)
        mt = sample_power_law_mass(rng, minmass[age_index], maxmass[age_index], imf)
        iso = isos[age_index]
        mass = np.asarray(iso.i_mass, dtype=float)
        k1 = np.interp(mt, mass, np.asarray(iso.v, dtype=float))
        j1 = np.interp(mt, mass, np.asarray(iso.b, dtype=float))

        xpix = rng.random() * npix
        ypix = rng.random() * npix
        dred = rng.normal() * sigred
        j_obs = j1 + (a_k + dred) * (1.25 / 2.2) ** (-1.7) + dmod
        k_obs = k1 + (a_k + dred) + dmod
        rows.append((xpix, ypix, j_obs, k_obs))

        lum1 += 10 ** (-0.4 * k1)
        count += 1

    rows = np.asarray(rows)
    with open(fileout, 'w') as f:
        f.write(f'{"X":>10}{"Y":>10}{"J":>10}{"K":>10}\n')
        np.savetxt(f, rows, fmt='%10.3f%10.3f%10.4f%10.4f')
    return rows
